# Case Study: Collatz

Let the Collatz transformation function $f: \mathbb{N}^+ \to \mathbb{N}^+$ be defined as:

$$f(n) = \begin{cases} \frac{n}{2} & \text{if } n \equiv 0 \pmod 2 \\ 3n + 1 & \text{if } n \equiv 1 \pmod 2 \end{cases}$$

Let $f^k(n)$ denote the $k$-fold composition of $f$ applied to $n$, where $f^0(n) = n$. 

The Collatz set $C$ is then defined as:

$$C = \{ n \in \mathbb{N}^+ \mid \exists k \in \mathbb{N} \text{ such that } f^k(n) = 1 \}$$

Let $t(n) = \min\{k \in \mathbb{N} \mid f^k(n) = 1\}$ be the stopping time, with $t(n) = \infty$ if no such $k$ exists.
Then $C = \{ n \in \mathbb{N}^+ \mid t(n) < \infty \}$, and the conjecture is $C = \mathbb{N}^+$.


In [ ]:
import dedekind # Check for library presence

## Pre-Processing

Transliteration of the above rules into executable code.

In [ ]:
# Legend for this draft of the surface:
#   exists today (Python):  𝔸, cond, π-arithmetic (% // * + ==), Σ, forall, first_where, η
#   exists in C++ only:     N * N | …, the projections and == on them, & | on terms (C++: && ||),
#                           the function certificates (IsFunctional ∧ IsEntire)
#   new:                    χ, the classifier under definition (Self node, Rec, the descent check),
#                           k - 1 on a term, the constant-graph leaf, from_cuts
#
# Pointwise logic on terms is & | ~ (Python cannot overload and / or / not); the same symbols on
# sets are the lattice.  & and | bind tighter than comparisons, so every comparison in a chain is
# parenthesised; a bare int where a term is expected raises and says so.

# ℕ⁺ as the positive part of the 64-bit word, ℕ's shadow:
N_64 = 𝔸(size_t) | n > 0
Nat = Annotated[int, N_64]                   # the checker sees int; the DSL reads the universe
# The budget is a coordinate of its own, in ℕ:
Dom = N_64 * 𝔸(size_t)                       # (n, k)

# The step, a function by structure (the guarded union of two function graphs):
f: Function[Nat, Nat] = N_64 * N_64 | (y == cond(x % 2 == 0, x // 2, 3*x + 1))

# "n reaches 1 within k steps", defined by recursion on k.  χ is the set's own characteristic
# arrow, mentioned inside its definition; the former closes the recursion, so the cases are
# joined at the term level, inside one former.
#   C(N, K) :- N = 1.                            converged; whatever budget remains is absorbed
#   C(N, K) :- N > 1, K > 0, f(N, M), C(M, K-1).   spend one unit of budget, step once
C: Relation[Nat, Nat] = Dom | ((n == 1) | ((n > 1) & (k > 0) & χ(f(n), k - 1)))
#   χ(f(n), k - 1) is C pulled back along ⟨f ∘ π1, π2 − 1⟩: f(n) substitutes the middle M,
#   because f is a function; k - 1 descends on a coordinate, so the recursion terminates.

# The verdict at budget B, read in K₃: ⊤ derived, U not yet, never ⊥.
χ_B: Function[Nat, K3] = N_64 * K3 | (y == Σ(C(x, B)))

## Evaluation Strategy

Problem conditioning for improved evaluation performance. E.g. adjustment of the domain or the computational budget. Dynamic programming and memoization.

In [ ]:
W, B = 1000, 178                               # the window on n, the budget on k
window = N_64 | n < W
assert forall(window, χ_B) == T                # a finite meet; 871 is the slowest seed, 178 steps
assert forall(window, χ_177) == U              # one step short: not yet decided
# Conditioning, not definition: the window bounds n, the budget bounds k, and because f is a
# function every χ(f(n), k - 1) is a substitution, not a search over a middle (the fusion law).
# Memoisation is the stage table C(·, k) reused for k + 1; compile time for small W, B, runtime otherwise.

## Post-Processing

E.g. classification by "escape velocity"

In [ ]:
t = stopping_time(C)                           # per n, the least k with C(n, k): first_where along k; "escape velocity"
early  = N_64 | (t(n) <= 100)                  # ⊤ within 100
late   = N_64 | (t(n) <= 200)                  # the next threshold
graded = from_cuts(early, late)                # ⊤ early, U late, ⊥ not within 200: all three values inhabited
assert graded(27) == U                         # t(27) = 111
# Σ's U means "not yet"; a grading's ⊥ means "not within this budget" and moves when you look harder.